Install dependencies

In [ ]:
# Install Hugging Face libraries
# !pip install -q transformers[torch] datasets sentencepiece accelerate

Use processed dataset and merge into one training set

In [2]:
import pandas as pd
from datasets import Dataset

# Load the two CSV files
df1 = pd.read_csv('data/processed/informal_formal.csv')
df2 = pd.read_csv('data/processed/seed_pairs.csv')

# Combine them into a single dataframe
# We drop duplicates in case the seed pairs are already in the informal_formal file
df_combined = pd.concat([df1, df2]).drop_duplicates().reset_index(drop=True)

print(f"Total training examples: {len(df_combined)}")

# T5 is a multi-task model, so we add a prefix to the input 
# so the model knows we are doing style transfer.
df_combined['informal'] = "transfer informal to formal: " + df_combined['informal']

# Convert to Hugging Face Dataset format
dataset = Dataset.from_pandas(df_combined)

# Split into Training (90%) and Validation (10%)
dataset = dataset.train_test_split(test_size=0.1)

Total training examples: 362


Tokenization using T5tokenizer

In [3]:
from transformers import T5Tokenizer

model_name = "t5-small" 
tokenizer = T5Tokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    # 1. Tokenize the informal text
    inputs = tokenizer(examples["informal"], max_length=128, truncation=True, padding="max_length")
    
    # 2. Tokenize the formal text (targets)
    targets = tokenizer(examples["formal"], max_length=128, truncation=True, padding="max_length")
    
    # 3. Replace tokenizer.pad_token_id (0) with -100 in the labels
    # Prevent the model from just learning to output padding
    labels = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] 
        for label in targets["input_ids"]
    ]
    
    inputs["labels"] = labels
    return inputs

# Re-run the mapping
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 37/37 [00:00<00:00, 6006.94 examples/s]


Using trainer API for training

In [4]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer

model = T5ForConditionalGeneration.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir="./formal_transformer_checkpoints",
    eval_strategy="epoch",  
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    num_train_epochs=5,     
    weight_decay=0.01,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 18100.34it/s]


Epoch,Training Loss,Validation Loss
1,No log,2.320987
2,No log,2.173681
3,No log,2.109361
4,No log,2.073110
5,No log,2.066122


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.71it/s]


TrainOutput(global_step=205, training_loss=2.2465241222846797, metrics={'train_runtime': 5.7541, 'train_samples_per_second': 282.405, 'train_steps_per_second': 35.626, 'total_flos': 54982606848000.0, 'train_loss': 2.2465241222846797, 'epoch': 5.0})

Inference and testing

In [10]:
def formalize(text):
    # Prepare the input with the task prefix
    input_text = "transfer informal to formal: " + text
    # Encode and generate
    inputs = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(
        inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )
    # Decode back to readable text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example sentences to formalize
examples = [
    "Tbh, I dunno if this plan is gonna work out.",
    "We're way behind schedule and stuff's getting messy.",
    "Just got a sick new phone, it's lit.",
    "Gonna skip the meeting, can't deal with it rn.",
    "That presentation was lowkey amazing, props to the team.",
    "We messed up big time and gotta fix it ASAP.",
    "She's super smart and totally crushed the interview.",
]

# Run all examples through the model
results = [(sentence, formalize(sentence)) for sentence in examples]

# Display as a formatted table
col_width = 55
header = f"{'Informal':<{col_width}} {'Formal':<{col_width}}"
divider = "-" * (col_width * 2 + 1)

print(divider)
print(header)
print(divider)
for informal, formal in results:
    print(f"{informal:<{col_width}} {formal:<{col_width}}")
print(divider)

---------------------------------------------------------------------------------------------------------------
Informal                                                Formal                                                 
---------------------------------------------------------------------------------------------------------------
Tbh, I dunno if this plan is gonna work out.            I am not sure if this plan is going to work out.       
We're way behind schedule and stuff's getting messy.    We are currently experiencing a significant delay in scheduling and the situation is experiencing considerable difficulty.
Just got a sick new phone, it's lit.                    I have recently purchased a new phone that has been upgraded.
Gonna skip the meeting, can't deal with it rn.          Gonna skip the meeting, and I am unable to deal with it.
That presentation was lowkey amazing, props to the team. That presentation was a pleasure, and I highly recommend it to the team.
We messed up